In [1]:
# Célula 1 — Imports
import geopandas as gpd
import pandas as pd
from pathlib import Path
from shapely.geometry import mapping

In [2]:
# Célula 2 — Carregar dados
pasta = Path(r"C:\Users\franc\OneDrive\Francisco\Profissional\MBA_Data_Science_ e_Analytics\00_TCC\06_Dados_base\GEO\2024_02_basegeo")

lf = gpd.read_file(pasta / "lf.shp")
bf = gpd.read_file(pasta / "bf.shp")

lf_val = lf[lf.geometry.notna() & lf.geometry.is_valid].copy()
bf_val = bf[bf.geometry.notna() & bf.geometry.is_valid].copy()

In [3]:
# Célula 3 — Calcular sobreposição dos 5 casos identificados
pares = [
    ("001636850000", "007071270000"),
    ("001636850000", "007071260000"),
    ("007025580000", "001811430000"),
    ("007025580000", "001811450000"),
    ("007025580000", "001208870000"),
]

print("Análise de sobreposição — 5 casos LF × BF")
print("=" * 80)
print(f"{'NUMBLOCO_LF':<15} {'AREA_LF':>10}  {'NUMBLOCO_BF':<15} {'AREA_BF':>10}  {'AREA_OVERLAP':>12}  {'% LF':>8}  {'% BF':>8}  {'RELACAO'}")
print("-" * 80)

for nb_lf, nb_bf in pares:
    lf_row = lf_val[lf_val["NUMBLOCO"] == nb_lf]
    bf_row = bf_val[bf_val["NUMBLOCO"] == nb_bf]

    if lf_row.empty or bf_row.empty:
        print(f"{nb_lf:<15} {'—':>10}  {nb_bf:<15} {'—':>10}  {'Não encontrado':>12}")
        continue

    geom_lf = lf_row.geometry.iloc[0]
    geom_bf = bf_row.geometry.iloc[0]
    area_lf = lf_row["AREA"].iloc[0]
    area_bf = bf_row["AREA"].iloc[0]

    if geom_lf.intersects(geom_bf):
        intersec = geom_lf.intersection(geom_bf)
        area_overlap = intersec.area
        pct_lf = area_overlap / geom_lf.area * 100
        pct_bf = area_overlap / geom_bf.area * 100

        if pct_bf >= 95:
            relacao = "BF DENTRO DE LF"
        elif pct_lf >= 95:
            relacao = "LF DENTRO DE BF"
        elif pct_bf >= 50:
            relacao = "BF MAJORITARIAMENTE EM LF"
        elif pct_lf >= 50:
            relacao = "LF MAJORITARIAMENTE EM BF"
        else:
            relacao = "SOBREPOSIÇÃO PARCIAL"
    else:
        area_overlap = 0
        pct_lf = 0
        pct_bf = 0
        relacao = "SEM INTERSEÇÃO REAL"

    print(f"{nb_lf:<15} {area_lf:>10.1f}  {nb_bf:<15} {area_bf:>10.1f}  {area_overlap:>12.1f}  {pct_lf:>7.1f}%  {pct_bf:>7.1f}%  {relacao}")

Análise de sobreposição — 5 casos LF × BF
NUMBLOCO_LF        AREA_LF  NUMBLOCO_BF        AREA_BF  AREA_OVERLAP      % LF      % BF  RELACAO
--------------------------------------------------------------------------------
001636850000       38232.3  007071270000        1298.5           0.4      0.0%      0.0%  SOBREPOSIÇÃO PARCIAL
001636850000       38232.3  007071260000        1310.6           3.2      0.0%      0.2%  SOBREPOSIÇÃO PARCIAL
007025580000        8035.0  001811430000         612.1           3.9      0.0%      0.6%  SOBREPOSIÇÃO PARCIAL
007025580000        8035.0  001811450000         788.5           5.3      0.1%      0.7%  SOBREPOSIÇÃO PARCIAL
007025580000        8035.0  001208870000         599.9           2.3      0.0%      0.4%  SOBREPOSIÇÃO PARCIAL
